# 世界の地震マップ — マグニチュードで見る地震の分布

**対象者：** 地震の発生場所や規模を、地図から直感的に知りたい高校生・大学生などの初学者。

## 改良した点と理由
1. **地図の背景を明るい CartoDB Positron に変更：** 地震の丸印が見やすくなるようにしました。
2. **ピン型マーカーを円形マーカーに変更：** 地震の位置を点として比較しやすくしました。
3. **マグニチュードに応じて円の色と大きさを変更：** 規模の違いを地図上で見分けやすくしました。大きい地震ほど円を大きくし、色も段階的に変えています。

**データ出典・取得日：** 2026年10月7日時点のUSGSデータ


In [1]:
from pathlib import Path
import pandas as pd
import folium
from IPython.display import display

# Notebookと同じフォルダーにあるCSVを読み込む
csv_path = Path("eq.csv")
if not csv_path.exists():
    raise FileNotFoundError("eq.csv をこのNotebookと同じフォルダーに置いてください。")

eq = pd.read_csv(csv_path)

# 地図に必要な列を数値に変換し、位置やマグニチュードが欠けた行を除外
for col in ["latitude", "longitude", "mag"]:
    eq[col] = pd.to_numeric(eq[col], errors="coerce")
eq = eq.dropna(subset=["latitude", "longitude", "mag"])

print(f"地図に使用する地震データ：{len(eq):,}件")
print(f"マグニチュード：{eq['mag'].min():.2f} ～ {eq['mag'].max():.2f}")
print(f"平均マグニチュード：{eq['mag'].mean():.2f}")

地図に使用する地震データ：1,850件
マグニチュード：2.45 ～ 6.60
平均マグニチュード：3.69


In [2]:
# 世界地図を作成（明るい背景地図を使用）
earthquake_map = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="CartoDB positron",
    control_scale=True
)

def magnitude_style(magnitude):
    # マグニチュードの大きさに応じて色を分ける
    if magnitude < 3:
        color = "#2b83ba"   # 青
    elif magnitude < 4:
        color = "#abdda4"   # 緑
    elif magnitude < 5:
        color = "#fdae61"   # オレンジ
    else:
        color = "#d7191c"   # 赤
    # 大きい地震ほど円を大きくする
    radius = 3 + magnitude * 1.3
    return color, radius

for _, row in eq.iterrows():
    magnitude = float(row["mag"])
    color, radius = magnitude_style(magnitude)
    place = str(row.get("place", "場所情報なし"))
    time = str(row.get("time", "時刻情報なし"))
    folium.CircleMarker(
        location=[float(row["latitude"]), float(row["longitude"])],
        radius=radius,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.65,
        tooltip=f"規模 M{magnitude:.2f}",
        popup=f"<b>マグニチュード：</b>M{magnitude:.2f}<br>"
              f"<b>場所：</b>{place}<br><b>時刻：</b>{time}"
    ).add_to(earthquake_map)

# 色の意味を示す凡例
legend_html = '''
<div style="
position: fixed; bottom: 25px; left: 25px; z-index: 9999;
background: white; padding: 10px 12px; border: 1px solid #888;
border-radius: 5px; font-size: 13px;">
<b>マグニチュード</b><br>
<span style="color:#2b83ba;">●</span> M3未満<br>
<span style="color:#abdda4;">●</span> M3以上～M4未満<br>
<span style="color:#fdae61;">●</span> M4以上～M5未満<br>
<span style="color:#d7191c;">●</span> M5以上<br>
<small>円が大きいほど規模が大きい</small>
</div>
'''
earthquake_map.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(earthquake_map)
display(earthquake_map)

## 地図から分かること・動作確認

- このCSVでは、マグニチュードは **2.45～6.60** の範囲でした。平均は約 **3.69** です。
- 地図を見ると、地震は世界中に分布しており、地域によって点が集まっている場所があります。円の色と大きさを使うことで、発生場所だけでなく規模の違いも見比べられます。
- **動作確認：** CSVを読み込み、緯度・経度・マグニチュードが欠けている行を除外してから地図を作成する処理にしました。地図上の円にマウスを合わせると規模が表示され、クリックすると場所と時刻も確認できます。実行時に表示された件数・最小値・最大値もCSVの内容と照合できます。

※ 地図上の点は、CSVに記録された地震を示します。色分けの境界は見やすさのために設定したもので、危険度を直接判定するものではありません。
